# European Flight Delay Prediction — Downsampling

This notebook downsamples the raw data to smaller sample which we will use for this project

## Prerequisites

 The project dependencies are defined in the repository-level `requirements.txt`
file. When this notebook is opened in Google Colab from GitHub, the repository is cloned into the Colab environment
before installing the required dependencies.

In [1]:
import os

# Clone the project repository into the Colab environment
REPO_URL = "https://github.com/sharonjmaye/european-flight-delay-prediction.git"
REPO_DIR = "european-flight-delay-prediction"
if os.path.exists(REPO_DIR):
    print(f"Repository already exists: {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}
    print("Repository cloned successfully.")

# Change the current working directory to the project root
%cd {REPO_DIR}

Cloning into 'european-flight-delay-prediction'...
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 7 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (7/7), done.
Resolving deltas: 100% (1/1), done.
Repository cloned successfully.
/content/european-flight-delay-prediction


## 1. Setup & Environment Installation

The project dependencies should be listed in `requirements.txt`. Run the following cell before continuing. If packages are newly installed, restart the Jupyter kernel if prompted.

In [2]:
%pip install -r requirements.txt

## 2. Configuration & GCP Authentication

Set the project and bucket that the pipeline will use & authenticate to GCP.

In [3]:
# --- GCP Configuration ---
PROJECT_ID = "setu-data-handling"
BUCKET_NAME = "european-flight-delay-prediction"
LOCATION = "EU"
# Source data  on Hugging Fataseace
RAW_URL = (
    "https://huggingface.co/datasets/345rf4gt56t4r3e3/"
    "flight-delays-europe-2023-2025/resolve/main/"
    "merged_flights_with_delay_2023_2025.parquet"
)

# Destination inside your GCS bucket
GCS_RAW_PATH = (
    "gs://european-flight-delay-prediction/"
    "raw/merged_flights_with_delay_2023_2025.parquet"
)

from google.colab import auth
auth.authenticate_user()

print(f"Target project: {PROJECT_ID}")
print(f"Target bucket: gs://{BUCKET_NAME}")

Target project: setu-data-handling
Target bucket: gs://european-flight-delay-prediction


## 3. Read Raw Parquet file and get schema details

In [4]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(GCS_RAW_PATH)

print(parquet_file.schema)

required group field_id=-1 schema {
  optional int64 field_id=-1 id (Int(bitWidth=64, isSigned=false));
  optional binary field_id=-1 icao24 (String);
  optional binary field_id=-1 flt_id (String);
  optional int64 field_id=-1 dof (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional binary field_id=-1 adep (String);
  optional binary field_id=-1 ades (String);
  optional binary field_id=-1 adep_p (String);
  optional binary field_id=-1 ades_p (String);
  optional binary field_id=-1 registration (String);
  optional binary field_id=-1 model (String);
  optional binary field_id=-1 typecode (String);
  optional binary field_id=-1 icao_aircraft_class (String);
  optional binary field_id=-1 icao_operator (String);
  optional int64 field_id=-1 first_seen (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 last_seen (Tim

## 3. Read Columns required for sampling and validation

In [5]:
import pandas as pd

columns = [
    "id",
    "source_year",
    "source_month",
    "delayed_15min",
    "adep",
    "ades",
    "icao_operator",
    "model",
    "typecode",
    "icao_aircraft_class",
    "route"
]

df = pd.read_parquet(
    GCS_RAW_PATH,
    columns=columns
)

## 4. Profile the raw data

In [6]:
print(f"Total rows: {len(df):,}")
print()

print("Year distribution:")
print(
    df["source_year"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)


print("\nDelay by year:")
delay_by_year = pd.crosstab(
    df["source_year"],
    df["delayed_15min"],
    normalize="index"
).mul(100).round(2)

delay_by_year.columns = [
    "not_delayed_pct",
    "delayed_pct"
]

print(delay_by_year)

print("\nDelay distribution:")
print(
    df["delayed_15min"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nUnique departure airports:", df["adep"].nunique())
print("\nUnique destination airports:", df["ades"].nunique())
print("\nUnique operators:", df["icao_operator"].nunique())
print("\nUnique aircraft models:", df["model"].nunique())
print("\nUnique routes:", df["route"].nunique())

print(f"\nTotal rows: {len(df):,}")
print(f"\nUnique IDs: {df['id'].nunique():,}")
print(f"\nMissing IDs: {df['id'].isna().sum():,}")
print(f"\nDuplicate IDs: {df['id'].duplicated().sum():,}")

missing = (
    df.isna()
      .sum()
      .to_frame("missing_count")
)

missing["missing_pct"] = (
    missing["missing_count"] / len(df) * 100
).round(2)

missing

Total rows: 15,300,770

Year distribution:
source_year
2023    30.03
2024    33.50
2025    36.46
Name: proportion, dtype: float64

Delay by year:
             not_delayed_pct  delayed_pct
source_year                              
2023                   93.03         6.97
2024                   92.57         7.43
2025                   89.63        10.37

Delay distribution:
delayed_15min
0    91.64
1     8.36
Name: proportion, dtype: float64

Unique departure airports: 991

Unique destination airports: 1075

Unique operators: 915

Unique aircraft models: 5932

Unique routes: 120157

Total rows: 15,300,770

Unique IDs: 15,300,770

Missing IDs: 0

Duplicate IDs: 0


,missing_count,missing_pct
id,0,0.00
source_year,0,0.00
source_month,0,0.00
delayed_15min,0,0.00
adep,0,0.00
ades,0,0.00
icao_operator,5963453,38.97
model,4636005,30.30
typecode,592662,3.87
icao_aircraft_class,2259233,14.77


## 4. Sampling configuration

In [7]:
SAMPLE_SIZE = 100_000
RANDOM_STATE = 42

STRATA_COLUMNS = [
    "source_month",
    "delayed_15min"
]

## 5. Calculate Population Strata


In [8]:
# Count records in each year-month / delay-class combination
strata = (
    df.groupby(STRATA_COLUMNS, dropna=False) # Sorts flights into groups eg 202301 + Not Delayed, 202301 + Delayed etc for each source_month
      .size() # Counts how many in each of these groups
      .reset_index(name="population_count") # truns the result into a table with count called population_count
)

# Work out what percentage each group is in whole raw dataset i.e. what share of the full dataset each year-month/delay combination represents,
# so we can preserve that same share in our 100,000-row sample.
strata["population_proportion"] = (
    strata["population_count"] / len(df)
)

strata.head()

,source_month,delayed_15min,population_count,population_proportion
0,202301,0,249171,0.016285
1,202301,1,20298,0.001327
2,202302,0,295439,0.019309
3,202302,1,24158,0.001579
4,202303,0,337439,0.022054


## 5. Calculate how many records to sample from each stratum

Using largest-remainder allocation to keep the sample proportional while guaranteeing exactly 100,000 rows




In [10]:
import numpy as np

# Calculate the exact proportional allocation
strata["exact_sample_size"] = (
    strata["population_proportion"] * SAMPLE_SIZE
)

# Initially allocate the integer part, rounding down
strata["sample_size"] = (
    np.floor(strata["exact_sample_size"])
    .astype(int)
)

# Calculate the fractional remainder
strata["remainder"] = (
    strata["exact_sample_size"] - strata["sample_size"]
)

# Determine how many records remain to reach exactly SAMPLE_SIZE
remaining = SAMPLE_SIZE - strata["sample_size"].sum()

# Allocate remaining records to strata with the largest remainders
if remaining > 0:
  largest_remainders = (
    strata["remainder"]
    .nlargest(remaining)
    .index
  )

strata.loc[largest_remainders, "sample_size"] += 1

print(f"Population size: {len(df):,}")
print(f"Target sample size: {SAMPLE_SIZE:,}")
print(f"Allocated sample size: {strata['sample_size'].sum():,}")
print(f"Number of strata: {len(strata):}")

Population size: 15,300,770
Target sample size: 100,000
Allocated sample size: 100,000
Number of strata: 72


## 6. Inspect the allocation before sampling



In [11]:
strata["population_pct"] = (
    strata["population_proportion"] * 100
).round(3)

strata[
    [
        "source_month",
        "delayed_15min",
        "population_count",
        "population_pct",
        "sample_size"
    ]
]

,source_month,delayed_15min,population_count,population_pct,sample_size
0,202301,0,249171,1.628,1628
1,202301,1,20298,0.133,133
2,202302,0,295439,1.931,1931
3,202302,1,24158,0.158,158
4,202303,0,337439,2.205,2205
...,...,...,...,...,...
67,202510,1,43113,0.282,282
68,202511,0,355571,2.324,2324
69,202511,1,38656,0.253,253
70,202512,0,377628,2.468,2468


## 7. Draw the random sample


In [12]:
sample_parts = []

for _, stratum in strata.iterrows():

    year_month = stratum["source_month"]
    delay_class = stratum["delayed_15min"]
    n = int(stratum["sample_size"])

    # Select records belonging to this stratum
    group = df[
        (df["source_month"] == year_month) &
        (df["delayed_15min"] == delay_class)
    ]

    # Randomly sample the required number of records
    group_sample = group.sample(
        n=n,
        random_state=RANDOM_STATE,
        replace=False
    )

    sample_parts.append(group_sample)

In [13]:
# Combine all the individual samples
sample_df = pd.concat(
    sample_parts,
    ignore_index=True
)

## 8. Shuffle the final dataset as it will currently be ordered by source_month

In [14]:
sample_df = (
    sample_df
    .sample(
        frac=1,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

## 9. Perform basic verification and store the ids we are using in our sample




In [15]:
assert len(sample_df) == SAMPLE_SIZE

print(f"Final sample size: {len(sample_df):,}")

sample_delay_distribution = (
    sample_df["delayed_15min"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nSample delay distribution:")
print(sample_delay_distribution)

sample_year_distribution = (
    sample_df["source_year"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nSample year distribution:")
print(sample_year_distribution)

assert len(sample_df) == 100_000

sample_ids = set(sample_df["id"])

Final sample size: 100,000

Sample delay distribution:
delayed_15min
0    91.64
1     8.36
Name: proportion, dtype: float64

Sample year distribution:
source_year
2023    30.03
2024    33.51
2025    36.46
Name: proportion, dtype: float64


## 10. Validate every stratum

Ensure there is the expected amount from each strata


In [16]:
sample_strata = (
    sample_df.groupby(STRATA_COLUMNS)
             .size()
             .reset_index(name="actual_sample_count")
)

strata_validation = strata.merge(
    sample_strata,
    on=STRATA_COLUMNS,
    how="left"
)

strata_validation["actual_sample_count"] = (
    strata_validation["actual_sample_count"]
    .fillna(0)
    .astype(int)
)

strata_validation["count_difference"] = (
    strata_validation["actual_sample_count"]
    - strata_validation["sample_size"]
)

strata_validation[
    [
        "source_month",
        "delayed_15min",
        "population_count",
        "sample_size",
        "actual_sample_count",
        "count_difference"
    ]
]

,source_month,delayed_15min,population_count,sample_size,actual_sample_count,count_difference
0,202301,0,249171,1628,1628,0
1,202301,1,20298,133,133,0
2,202302,0,295439,1931,1931,0
3,202302,1,24158,158,158,0
4,202303,0,337439,2205,2205,0
...,...,...,...,...,...,...
67,202510,1,43113,282,282,0
68,202511,0,355571,2324,2324,0
69,202511,1,38656,253,253,0
70,202512,0,377628,2468,2468,0


In [17]:
assert (strata_validation["count_difference"] == 0).all()

print("\nAll strata sampled as expected.")


All strata sampled as expected.


## 11. Capture the selected Id's and ensure there are 100,000 of them


In [18]:
# Store the IDs of the 100,000 selected flights

sample_ids = set(sample_df["id"].tolist())

print(f"Selected IDs: {len(sample_ids):,}")

assert len(sample_ids) == SAMPLE_SIZE, (
    "Number of unique sample IDs does not match SAMPLE_SIZE."
)

Selected IDs: 100,000


## 12. Retrieve the complete raw records for the selected IDs

The sampling dataframe contains only the columns needed for profiling, sampling and validation.
The 100,000 selected `id` values are now used to retrieve the **complete original records**
from the raw Parquet dataset.

The raw Parquet file is read in batches so that all 15.3 million complete records do not
need to be loaded into memory at the same time.


In [19]:
# Reuse the raw Parquet file opened earlier
# and scan it in manageable batches.

BATCH_SIZE = 250_000
matched_batches = []

for batch in parquet_file.iter_batches(batch_size=BATCH_SIZE):
    batch_df = batch.to_pandas()

    # Keep only complete raw records whose ID was selected in the sample
    matched = batch_df[batch_df["id"].isin(sample_ids)]

    if not matched.empty:
        matched_batches.append(matched)

# Combine the matching records into the final full sample
full_sample_df = pd.concat(
    matched_batches,
    ignore_index=True
)

print(f"Complete sample records retrieved: {len(full_sample_df):,}")
print(f"Columns retrieved: {full_sample_df.shape[1]}")


Complete sample records retrieved: 100,000
Columns retrieved: 24


## 13. Validate the complete sample

Confirm that:

- exactly 100,000 complete records were retrieved;
- every retrieved ID belongs to the stratified sample;
- no selected IDs are missing; and
- all columns from the raw Parquet dataset have been retained.


In [20]:
# Validate record count
assert len(full_sample_df) == SAMPLE_SIZE, (
    f"Expected {SAMPLE_SIZE:,} records, "
    f"but retrieved {len(full_sample_df):,}."
)

# Validate IDs
retrieved_ids = set(full_sample_df["id"].tolist())

assert retrieved_ids == sample_ids, (
    "Retrieved IDs do not exactly match the selected sample IDs."
)

# Validate that all raw columns were retained
raw_columns = parquet_file.schema.names

assert set(full_sample_df.columns) == set(raw_columns), (
    "The processed sample does not contain the same columns as the raw dataset."
)

print("Complete sample validation passed.")
print(f"Rows: {full_sample_df.shape[0]:,}")
print(f"Columns: {full_sample_df.shape[1]}")
print(f"Unique IDs: {full_sample_df['id'].nunique():,}")


Complete sample validation passed.
Rows: 100,000
Columns: 24
Unique IDs: 100,000


## 14. Re-check the key sample distributions

The complete records should retain the same delay and year distributions already validated
for the stratified sample.


In [21]:
processed_delay_distribution = (
    full_sample_df["delayed_15min"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

processed_year_distribution = (
    full_sample_df["source_year"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("Processed sample delay distribution:")
print(processed_delay_distribution)

print("\nProcessed sample year distribution:")
print(processed_year_distribution)


Processed sample delay distribution:
delayed_15min
0    91.64
1     8.36
Name: proportion, dtype: float64

Processed sample year distribution:
source_year
2023    30.03
2024    33.51
2025    36.46
Name: proportion, dtype: float64


## 15. Save the processed sample locally as Parquet

The processed dataset retains all original columns for the 100,000 selected flights.
Parquet is retained as the processed file format because it preserves data types and provides
efficient compressed storage.


In [22]:
LOCAL_PROCESSED_PATH = "/content/flights_downsampled_100k.parquet"

# Shuffle the complete records before saving.
# This changes only row order, not the selected records.
full_sample_df = (
    full_sample_df
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

full_sample_df.to_parquet(
    LOCAL_PROCESSED_PATH,
    index=False,
    engine="pyarrow"
)

processed_size_mb = os.path.getsize(LOCAL_PROCESSED_PATH) / (1024 * 1024)

print(f"Processed dataset saved locally: {LOCAL_PROCESSED_PATH}")
print(f"File size: {processed_size_mb:.2f} MB")


Processed dataset saved locally: /content/flights_downsampled_100k.parquet
File size: 6.41 MB


## 16. Upload the processed sample to Google Cloud Storage

The final processed Parquet file is stored in the `processed/` folder of the project bucket:

`gs://european-flight-delay-prediction/processed/flights_downsampled_100k.parquet`


In [23]:
from google.cloud import storage

GCS_PROCESSED_OBJECT = "processed/flights_downsampled_100k.parquet"

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(GCS_PROCESSED_OBJECT)

blob.upload_from_filename(
    LOCAL_PROCESSED_PATH,
    content_type="application/octet-stream"
)

print(
    f"Uploaded processed dataset to "
    f"gs://{BUCKET_NAME}/{GCS_PROCESSED_OBJECT}"
)


Uploaded processed dataset to gs://european-flight-delay-prediction/processed/flights_downsampled_100k.parquet


## 17. Verify the GCS upload

Confirm that the processed object exists in GCS and that its stored size matches the local
Parquet file.


In [24]:
blob.reload()

local_size = os.path.getsize(LOCAL_PROCESSED_PATH)
gcs_size = blob.size

assert gcs_size == local_size, (
    f"Size mismatch: local file is {local_size:,} bytes "
    f"but GCS object is {gcs_size:,} bytes."
)

print("GCS upload verification passed.")
print(f"GCS path: gs://{BUCKET_NAME}/{GCS_PROCESSED_OBJECT}")
print(f"Stored size: {gcs_size / (1024 * 1024):.2f} MB")
print(f"Generation: {blob.generation}")


GCS upload verification passed.
GCS path: gs://european-flight-delay-prediction/processed/flights_downsampled_100k.parquet
Stored size: 6.41 MB
Generation: 1791128434018298
